In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
import pandas as pd

data = fetch_20newsgroups(subset='all', categories=['sci.med', 'sci.space'], remove=('headers', 'footers', 'quotes'))
df = pd.DataFrame({'text': data.data, 'label': data.target})
print(df.shape)
print(df['label'].value_counts())

(1977, 2)
label
0    990
1    987
Name: count, dtype: int64


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['sci.med', 'sci.space']))

              precision    recall  f1-score   support

     sci.med       0.91      0.92      0.91       193
   sci.space       0.92      0.92      0.92       203

    accuracy                           0.92       396
   macro avg       0.92      0.92      0.92       396
weighted avg       0.92      0.92      0.92       396



In [3]:
# What words most strongly predict each class?
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

import numpy as np
top_space = np.argsort(coefficients)[-10:]
top_med = np.argsort(coefficients)[:10]

print("Top words → sci.space:")
print([feature_names[i] for i in top_space])

print("\nTop words → sci.med:")
print([feature_names[i] for i in top_med])

Top words → sci.space:
['spacecraft', 'sky', 'launch', 'shuttle', 'earth', 'nasa', 'orbit', 'moon', 'the', 'space']

Top words → sci.med:
['my', 'medical', 'msg', 'she', 'your', 'is', 'doctor', 'disease', 'me', 'cancer']


TF-IDF
Converts text into numeric vectors using word frequency, penalizing words that appear across all documents. No neural network — pure statistics. Each document becomes a fixed-size vector of word importance scores.
Why fit only on train
fit learns the vocabulary. Fitting on test data leaks information — the vectorizer would adapt to test words, making evaluation unreliable. Fit on train, transform everything else.
Stopwords problem
Top predictors for sci.med included my, she, your, is, me — meaningless filler words. The model partially learned conversational tone instead of medical content. Remove stopwords before vectorizing to force the model to learn real signal.
Classical NLP ceiling vs embeddings
TF-IDF has no understanding of meaning — cancer and tumor are unrelated to it. nasa and space agency are completely different vectors. Distilbert breaks this ceiling because semantically similar words cluster close together in vector space, learned from billions of examples.

In [5]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model_bert = AutoModel.from_pretrained("distilbert-base-uncased")
model_bert.eval()

def get_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad():
            output = model_bert(**encoded)
        embeddings = output.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(embeddings)
    return np.vstack(all_embeddings)

print("Generating embeddings...")
X_train_bert = get_embeddings(X_train.reset_index(drop=True))
X_test_bert = get_embeddings(X_test.reset_index(drop=True))
print("Done. Shape:", X_train_bert.shape)

Generating embeddings...
Done. Shape: (1581, 768)


In [6]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_bert, y_train)

y_pred_bert = model_lr.predict(X_test_bert)
print(classification_report(y_test, y_pred_bert, target_names=['sci.med', 'sci.space']))

              precision    recall  f1-score   support

     sci.med       0.94      0.91      0.92       193
   sci.space       0.92      0.94      0.93       203

    accuracy                           0.93       396
   macro avg       0.93      0.93      0.93       396
weighted avg       0.93      0.93      0.93       396



[CLS] token vs mean pooling — when to use which
Why distilbert only marginally outperformed TF-IDF here
Always run classical baseline before reaching for transformers

[CLS] token vs mean pooling
Use [CLS] for classification tasks — distilbert was pretrained with [CLS] designed to represent the whole sentence for this purpose. Use mean pooling for similarity tasks where the model hasn't been fine-tuned, as averaging all token embeddings captures meaning more reliably than a single token.
Why distilbert only marginally outperformed TF-IDF
sci.med and sci.space use completely different vocabulary — word frequency alone is nearly sufficient to separate them. Distilbert's semantic understanding only pulls ahead when categories share vocabulary, text is ambiguous, or context/negation matters. Easy datasets don't expose the gap.
Always run classical baseline first
TF-IDF + Logistic Regression is fast, interpretable, and often good enough. If distilbert gives you 1% gain over TF-IDF, the added complexity, compute cost, and inference time may not be worth it. Reach for transformers only when the baseline demonstrably fails.

In [7]:
# Find misclassified examples
X_test_list = X_test.reset_index(drop=True)
wrong_idx = np.where(y_pred_bert != y_test.reset_index(drop=True))[0]

print(f"Total misclassified: {len(wrong_idx)}\n")
for i in wrong_idx[:5]:
    print(f"True: {data.target_names[y_test.reset_index(drop=True)[i]]}")
    print(f"Predicted: {data.target_names[y_pred_bert[i]]}")
    print(f"Text snippet: {X_test_list[i][:200]}")
    print("---")

Total misclassified: 29

True: sci.med
Predicted: sci.space
Text snippet: 



---
True: sci.med
Predicted: sci.space
Text snippet: 
: ... I think they should rename Waco TX to Wacko TX!
---
True: sci.med
Predicted: sci.space
Text snippet: I have become involved in a project to further develop and 
improve the performance of SPECT (Single Photon Emission
Computerized Tomography) imaging.  We will eventually have
to peddle this stuff som
---
True: sci.space
Predicted: sci.med
Text snippet: If available please send to
Glen Moore
Director
Science Centre
Wollongong, Australia
fax: 61 42 213151   email: gkm@cc.uow.edu.au


---
True: sci.med
Predicted: sci.space
Text snippet: *****************************************************************************
*                  Lyme Disease Electronic Mail Network                     *
*                          LymeNet Newslette
---


This is exactly what error analysis is for. Look at what you're seeing:
Empty/near-empty texts — first two examples have no content. The model is guessing on noise. These should have been caught in cleaning.
Ambiguous content — "Waco TX" misclassified as sci.space. That text has no medical content — it's a political comment that leaked into a medical newsgroup. The model had nothing real to work with.
Cross-domain vocabulary — SPECT (medical imaging) contains technical terms that possibly overlap with space/physics vocabulary. The model got confused by domain overlap.
No content, just metadata — the Australia example is a contact/address block with no topical content. Again, cleaning failure.
The pattern: most errors aren't the model being dumb — they're garbage input. This is why data cleaning is unglamorous but critical. A better cleaning step would have removed empty texts and near-empty texts before training.


Error analysis always follows evaluation — a single metric hides where the model fails
Most misclassifications trace back to data quality, not model quality
Empty/short texts should be filtered before training